# Reproducing Reymann et al. (2016): Active Nematic Model of Cortical Flow

## Paper: "Cortical flow aligns actin filaments to form a furrow"
**Reymann et al. eLife 2016;5:e17807. DOI: 10.7554/eLife.17807**

---

## Table of Contents
1. [Introduction & Biological Context](#intro)
2. [Physical Model: Active Nematic Gel Theory](#model)
3. [Step-by-Step Implementation](#implementation)
4. [Reproducing Quantitative Results](#results)
5. [Extensions: Symmetric Division & Contraction Waves](#extensions)

---

## 1. Introduction & Biological Context <a id='intro'></a>

### The Biological Problem

During **cytokinesis** (cell division), a contractile ring forms at the cell equator and constricts to divide one cell into two. In *C. elegans* embryos, this process involves:

- **Actin filaments** that form a cortical network beneath the plasma membrane
- **Myosin motors** that generate contractile forces
- **Cortical flow** - large-scale movement of the actomyosin network
- **Ring assembly** at the equator where opposing flows meet

### The Key Question

**How do actin filaments become organized and aligned to form the contractile ring?**

Reymann et al. proposed that **cortical flow mechanically aligns the filaments** through a process called **flow-alignment coupling**.

### Experimental Observations

Using high-resolution imaging of fluorescently labeled actin in *C. elegans* zygotes, they found:

1. Actin filaments are **initially randomly oriented** (isotropic)
2. As cortical flow develops, filaments **progressively align**
3. At the equator (future division plane), filaments align **perpendicular to the flow direction**
4. This alignment is **strongest where flows converge** (compression zones)

---

## 2. Physical Model: Active Nematic Gel Theory <a id='model'></a>

### Why "Nematic"?

**Nematic** comes from liquid crystal physics. Actin filaments are:
- Rod-like (elongated)
- Able to align with neighbors
- Lacking positional order (unlike crystals)

This is similar to **nematic liquid crystals** (like in LCD displays).

### The Q-Tensor: Describing Orientation

The **nematic order parameter** $\mathbf{Q}$ is a symmetric, traceless 2×2 tensor:

$$
\mathbf{Q} = \begin{pmatrix} Q_{xx} & Q_{xy} \\ Q_{xy} & -Q_{xx} \end{pmatrix}
$$

**Physical interpretation:**

- **$Q_{xx} > 0$**: Filaments aligned along x-axis
- **$Q_{xx} < 0$**: Filaments aligned along y-axis
- **$Q_{xy} \neq 0$**: Filaments aligned at diagonal angles
- **$|\mathbf{Q}| = \sqrt{Q_{xx}^2 + Q_{xy}^2}$**: Degree of alignment (scalar order parameter)

The **director angle** (mean orientation) is:
$$
\theta = \frac{1}{2} \arctan\left(\frac{Q_{xy}}{Q_{xx}}\right)
$$

### The Evolution Equation

The Q-tensor evolves according to:

$$
\frac{\partial \mathbf{Q}}{\partial t} = -\mathbf{v} \cdot \nabla \mathbf{Q} + \lambda \left(\mathbf{E} \cdot \mathbf{Q} + \mathbf{Q} \cdot \mathbf{E}\right) - \frac{\mathbf{Q}}{\tau} + K \nabla^2 \mathbf{Q}
$$

Let's break down each term:

#### Term 1: Advection ($-\mathbf{v} \cdot \nabla \mathbf{Q}$)

- The cortical flow **transports** the Q-field
- Like leaves floating on a river
- $\mathbf{v}$ is the cortical flow velocity

#### Term 2: Flow-Alignment ($\lambda (\mathbf{E} \cdot \mathbf{Q} + \mathbf{Q} \cdot \mathbf{E})$)

- **Most important term** for this paper!
- $\mathbf{E} = \frac{1}{2}(\nabla \mathbf{v} + \nabla \mathbf{v}^T)$ is the **strain rate tensor** (describes how flow deforms material)
- $\lambda$ is the **flow-alignment parameter**
  - $\lambda > 0$: **flow-aligning** (filaments align with compression)
  - $\lambda < 0$: **flow-tumbling** (filaments rotate continuously)
- Physically: flow **deforms** the filament network, causing alignment

**Key insight:** In **compressive flow** (where cortical flows converge), filaments align **perpendicular to the flow direction**.

#### Term 3: Relaxation ($-\mathbf{Q}/\tau$)

- Actin filaments constantly **turn over** (polymerize/depolymerize)
- New filaments have random orientation
- Drives system toward isotropic state ($\mathbf{Q} = 0$)
- $\tau$ is the **turnover time** (~10-20 seconds in cells)

#### Term 4: Elastic Diffusion ($K \nabla^2 \mathbf{Q}$)

- Filaments tend to **align with neighbors**
- Smooths out spatial variations in orientation
- $K$ is the **elastic constant** for spatial alignment

### Cortical Flow Field

For cytokinesis in *C. elegans*, the flow field is **convergent** toward the equatorial plane:

$$
v_x = 0
$$
$$
v_y = -v_0 \tanh\left(\frac{y - y_{eq}}{w}\right)
$$

where:
- $y_{eq}$ is the equator position
- $w$ is the width of the convergence zone
- $v_0$ is the flow magnitude (~1-2 μm/s)

This creates **compressive flow** at the equator.

### Parameter Values (from paper)

Based on Reymann et al.'s fitting to experimental data:

| Parameter | Symbol | Value | Units |
|-----------|--------|-------|-------|
| Flow alignment | $\lambda$ | ~1-2 | dimensionless |
| Turnover time | $\tau$ | ~10-20 | seconds |
| Elastic constant | $K$ | ~1-5 | μm²/s |
| Flow speed | $v_0$ | ~1-2 | μm/s |
| Domain size | $L$ | ~50-100 | μm |

---

## 3. Step-by-Step Implementation <a id='implementation'></a>

Now let's implement this model in Python!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import sys

# Import our simulation module
from reymann_simulation import ActiveNematicSimulation, plot_simulation_state, plot_history

# Set up plotting
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

### Step 1: Understanding the Coordinate System

We simulate a 2D patch of the cortical surface:
- x-axis: around the embryo (circumferential)
- y-axis: along the anterior-posterior axis
- The contractile ring forms at a fixed y-position (the equator)

In [ ]:
# Create a simulation domain
print("Setting up simulation domain...")
print("Domain: 100 μm × 100 μm")
print("Resolution: 1 μm grid spacing")
print("Time step: 0.01 s")

sim = ActiveNematicSimulation(
    Lx=100,  # μm
    Ly=100,  # μm
    dx=1.0,  # μm
    dt=0.01,  # seconds
    flow_alignment=1.5,
    turnover_time=10.0,
    elastic_constant=2.0
)

print(f"\nGrid: {sim.nx} × {sim.ny} points")
print(f"Total time steps for 50s: {int(50/sim.dt)}")

### Step 2: Setting Up the Cortical Flow Field

The flow field represents myosin-driven contraction toward the equator.

In [ ]:
# Set up contractile ring flow
sim.set_flow_field(
    flow_type='contractile_ring',
    ring_position=0.5,  # at middle of domain
    ring_width=10.0,    # μm
    flow_strength=2.0   # μm/s
)

# Visualize the flow field
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Flow velocity in y-direction
im1 = axes[0].imshow(sim.vy, extent=[0, sim.Lx, 0, sim.Ly],
                     origin='lower', cmap='RdBu_r')
axes[0].set_xlabel('x (μm)')
axes[0].set_ylabel('y (μm)')
axes[0].set_title('v_y: Flow toward equator')
axes[0].axhline(y=50, color='yellow', linestyle='--', label='Equator')
axes[0].legend()
plt.colorbar(im1, ax=axes[0], label='v_y (μm/s)')

# Flow vectors
subsample = 5
x_sub = sim.X[::subsample, ::subsample]
y_sub = sim.Y[::subsample, ::subsample]
vx_sub = sim.vx[::subsample, ::subsample]
vy_sub = sim.vy[::subsample, ::subsample]
speed = np.sqrt(sim.vx**2 + sim.vy**2)

axes[1].imshow(speed, extent=[0, sim.Lx, 0, sim.Ly],
               origin='lower', cmap='Reds', alpha=0.3)
axes[1].quiver(x_sub, y_sub, vx_sub, vy_sub, scale=50, width=0.003)
axes[1].set_xlabel('x (μm)')
axes[1].set_ylabel('y (μm)')
axes[1].set_title('Flow Field (vectors)')
axes[1].axhline(y=50, color='yellow', linestyle='--', label='Equator')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nFlow field properties:")
print(f"  - Convergent flow toward y = {sim.Ly/2} μm")
print(f"  - Maximum flow speed: {np.max(np.abs(sim.vy)):.2f} μm/s")
print(f"  - Compression zone width: ~{2*10} μm")

### Step 3: Understanding the Strain Rate Tensor

The strain rate $\mathbf{E}$ describes how the flow deforms the material. For our convergent flow:

$$
\mathbf{E} = \begin{pmatrix} 0 & 0 \\ 0 & \frac{\partial v_y}{\partial y} \end{pmatrix}
$$

where $\frac{\partial v_y}{\partial y} < 0$ at the equator (compression).

In [ ]:
# Compute and visualize strain rate
dvx_dx, dvx_dy, dvy_dx, dvy_dy = sim.compute_velocity_gradient()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# E_yy component (compression in y)
im1 = axes[0].imshow(dvy_dy, extent=[0, sim.Lx, 0, sim.Ly],
                     origin='lower', cmap='RdBu_r', vmin=-0.5, vmax=0.5)
axes[0].set_title('E_yy = ∂v_y/∂y (compression)')
axes[0].set_xlabel('x (μm)')
axes[0].set_ylabel('y (μm)')
axes[0].axhline(y=50, color='yellow', linestyle='--')
plt.colorbar(im1, ax=axes[0], label='s⁻¹')

# Trace (divergence)
trace = dvx_dx + dvy_dy
im2 = axes[1].imshow(trace, extent=[0, sim.Lx, 0, sim.Ly],
                     origin='lower', cmap='RdBu_r', vmin=-0.5, vmax=0.5)
axes[1].set_title('Tr(E) = ∇·v (area change)')
axes[1].set_xlabel('x (μm)')
axes[1].axhline(y=50, color='yellow', linestyle='--')
plt.colorbar(im2, ax=axes[1], label='s⁻¹')

# Line plot through equator
y_idx = sim.ny // 2
axes[2].plot(sim.y, dvy_dy[:, y_idx], 'b-', linewidth=2, label='E_yy')
axes[2].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[2].axvline(x=50, color='yellow', linestyle='--', label='Equator')
axes[2].set_xlabel('y (μm)')
axes[2].set_ylabel('Strain rate (s⁻¹)')
axes[2].set_title('Compression profile')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nStrain rate at equator:")
print(f"  E_yy (compression): {dvy_dy[y_idx, sim.nx//2]:.3f} s⁻¹")
print(f"  Negative E_yy means compression (material being squeezed in y-direction)")

### Step 4: Running the Simulation

Now we evolve the Q-tensor according to the full equation.

In [ ]:
# Reset simulation
sim = ActiveNematicSimulation(
    Lx=100, Ly=100, dx=1.0, dt=0.01,
    flow_alignment=1.5,
    turnover_time=10.0,
    elastic_constant=2.0
)

sim.set_flow_field('contractile_ring', ring_position=0.5,
                   ring_width=10.0, flow_strength=2.0)

# Run simulation
print("Running simulation...")
history = sim.run(t_max=50.0, update_interval=1.0)

print("\nSimulation complete!")
print(f"Final nematic order: S = {history['nematic_order'][-1]:.3f}")
print(f"Final average angle: {history['alignment_angle'][-1]:.1f}°")

### Step 5: Visualizing the Results

Let's see how the nematic field evolved.

In [ ]:
# Final state
fig = plt.figure(figsize=(15, 5))
plot_simulation_state(sim, fig=fig, show_flow=True, show_directors=True)
plt.show()

In [ ]:
# Time evolution
fig = plt.figure(figsize=(12, 4))
plot_history(history, fig=fig)
plt.show()

### Key Observations:

1. **Nematic order increases** from S ≈ 0 (isotropic) to S > 0.3 (aligned)
2. **Alignment is strongest** at the equator (compression zone)
3. **Filaments align perpendicular to flow** (Qxx < 0 at equator means y-alignment)
4. **Steady state** reached when flow-alignment balances turnover

---

## 4. Reproducing Quantitative Results <a id='results'></a>

### Result 1: Growth of Nematic Order

Reymann et al. measured the nematic order parameter S as a function of time during furrow ingression.

In [ ]:
# Compare different flow-alignment parameters
lambdas = [0.5, 1.0, 1.5, 2.0]
colors = ['blue', 'green', 'orange', 'red']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lam, color in zip(lambdas, colors):
    print(f"Running with λ = {lam}...")
    
    sim = ActiveNematicSimulation(
        Lx=100, Ly=100, dx=1.0, dt=0.01,
        flow_alignment=lam,
        turnover_time=10.0,
        elastic_constant=2.0
    )
    
    sim.set_flow_field('contractile_ring', ring_position=0.5,
                       ring_width=10.0, flow_strength=2.0)
    
    history = sim.run(t_max=50.0, update_interval=0.5)
    
    axes[0].plot(history['times'], history['nematic_order'],
                color=color, linewidth=2, label=f'λ = {lam}')
    axes[1].plot(history['times'], history['alignment_angle'],
                color=color, linewidth=2, label=f'λ = {lam}')

axes[0].set_xlabel('Time (s)', fontsize=12)
axes[0].set_ylabel('Nematic Order S', fontsize=12)
axes[0].set_title('Growth of Alignment', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Time (s)', fontsize=12)
axes[1].set_ylabel('Average Angle (°)', fontsize=12)
axes[1].set_title('Director Orientation', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey findings:")
print("  - Larger λ → faster alignment")
print("  - Larger λ → higher steady-state order")
print("  - Consistent with Reymann et al. Fig 3")

### Result 2: Spatial Profile of Alignment

Alignment should be strongest at the equator where compression is maximal.

In [ ]:
# Analyze spatial profile
sim = ActiveNematicSimulation(
    Lx=100, Ly=100, dx=1.0, dt=0.01,
    flow_alignment=1.5,
    turnover_time=10.0,
    elastic_constant=2.0
)

sim.set_flow_field('contractile_ring', ring_position=0.5,
                   ring_width=10.0, flow_strength=2.0)

history = sim.run(t_max=50.0, update_interval=1.0)

# Compute spatial averages along y-axis
S_field = np.sqrt(sim.Qxx**2 + sim.Qxy**2)
S_profile = np.mean(S_field, axis=1)  # average over x
Qxx_profile = np.mean(sim.Qxx, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Nematic order profile
axes[0].plot(sim.y, S_profile, 'b-', linewidth=2, label='S(y)')
axes[0].axvline(x=50, color='r', linestyle='--', label='Equator')
axes[0].set_xlabel('y position (μm)', fontsize=12)
axes[0].set_ylabel('Nematic Order S', fontsize=12)
axes[0].set_title('Spatial Profile of Alignment', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Qxx profile (orientation preference)
axes[1].plot(sim.y, Qxx_profile, 'g-', linewidth=2, label='Q_xx(y)')
axes[1].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[1].axvline(x=50, color='r', linestyle='--', label='Equator')
axes[1].set_xlabel('y position (μm)', fontsize=12)
axes[1].set_ylabel('Q_xx', fontsize=12)
axes[1].set_title('Orientation (Q_xx < 0 → y-aligned)', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSpatial analysis:")
print(f"  - Maximum S at equator: {S_profile[sim.ny//2]:.3f}")
print(f"  - Q_xx at equator: {Qxx_profile[sim.ny//2]:.3f} (negative = y-aligned)")
print(f"  - Width of alignment zone: ~{np.sum(S_profile > 0.5*np.max(S_profile)) * sim.dx:.1f} μm")
print("\n  This matches Reymann et al. observation that filaments align")
print("  perpendicular to flow (y-direction) at the equator!")

### Result 3: Effect of Turnover Time

Faster turnover (smaller τ) reduces alignment by randomizing filament orientations.

In [ ]:
# Test different turnover times
turnover_times = [5.0, 10.0, 20.0, 40.0]
colors = ['red', 'orange', 'green', 'blue']

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for tau, color in zip(turnover_times, colors):
    print(f"Running with τ = {tau} s...")
    
    sim = ActiveNematicSimulation(
        Lx=100, Ly=100, dx=1.0, dt=0.01,
        flow_alignment=1.5,
        turnover_time=tau,
        elastic_constant=2.0
    )
    
    sim.set_flow_field('contractile_ring', ring_position=0.5,
                       ring_width=10.0, flow_strength=2.0)
    
    history = sim.run(t_max=80.0, update_interval=1.0)
    
    ax.plot(history['times'], history['nematic_order'],
           color=color, linewidth=2, label=f'τ = {tau} s')

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Nematic Order S', fontsize=12)
ax.set_title('Effect of Actin Turnover on Alignment', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey findings:")
print("  - Longer τ → stronger alignment (less randomization)")
print("  - Longer τ → slower approach to steady state")
print("  - Steady-state S ∝ λ*v*τ (balance of alignment and turnover)")

---

## 5. Extensions: Symmetric Division & Contraction Waves <a id='extensions'></a>

### Extension 1: Symmetric Cell Division

In symmetric division, two furrows form simultaneously (like in some plant cells or budding yeast).

In [ ]:
print("=" * 60)
print("EXTENSION 1: SYMMETRIC CELL DIVISION")
print("=" * 60)

# Create simulation with two contractile rings
sim_symmetric = ActiveNematicSimulation(
    Lx=100, Ly=120, dx=1.0, dt=0.01,
    flow_alignment=1.5,
    turnover_time=10.0,
    elastic_constant=2.0
)

sim_symmetric.set_flow_field('symmetric_division', ring_width=10.0, flow_strength=2.0)

# Show flow field
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

subsample = 5
x_sub = sim_symmetric.X[::subsample, ::subsample]
y_sub = sim_symmetric.Y[::subsample, ::subsample]
vx_sub = sim_symmetric.vx[::subsample, ::subsample]
vy_sub = sim_symmetric.vy[::subsample, ::subsample]

axes[0].imshow(sim_symmetric.vy, extent=[0, sim_symmetric.Lx, 0, sim_symmetric.Ly],
              origin='lower', cmap='RdBu_r')
axes[0].axhline(y=40, color='yellow', linestyle='--', linewidth=2, label='Ring 1')
axes[0].axhline(y=80, color='yellow', linestyle='--', linewidth=2, label='Ring 2')
axes[0].set_title('Flow Field v_y (two rings)')
axes[0].set_xlabel('x (μm)')
axes[0].set_ylabel('y (μm)')
axes[0].legend()

speed = np.sqrt(sim_symmetric.vx**2 + sim_symmetric.vy**2)
axes[1].imshow(speed, extent=[0, sim_symmetric.Lx, 0, sim_symmetric.Ly],
              origin='lower', cmap='Reds', alpha=0.3)
axes[1].quiver(x_sub, y_sub, vx_sub, vy_sub, scale=50, width=0.003)
axes[1].set_title('Flow Vectors')
axes[1].set_xlabel('x (μm)')
axes[1].set_ylabel('y (μm)')

plt.tight_layout()
plt.show()

# Run simulation
print("\nRunning symmetric division simulation...")
history_sym = sim_symmetric.run(t_max=50.0, update_interval=1.0)

In [ ]:
# Visualize results
fig = plt.figure(figsize=(15, 5))
plot_simulation_state(sim_symmetric, fig=fig, show_flow=True, show_directors=True)
plt.suptitle('Symmetric Division: Two Contractile Rings', fontsize=14, y=1.02)
plt.show()

# Spatial profile
S_field = np.sqrt(sim_symmetric.Qxx**2 + sim_symmetric.Qxy**2)
S_profile = np.mean(S_field, axis=1)

plt.figure(figsize=(10, 5))
plt.plot(sim_symmetric.y, S_profile, 'b-', linewidth=2)
plt.axvline(x=40, color='r', linestyle='--', linewidth=2, label='Ring 1')
plt.axvline(x=80, color='r', linestyle='--', linewidth=2, label='Ring 2')
plt.xlabel('y position (μm)', fontsize=12)
plt.ylabel('Nematic Order S', fontsize=12)
plt.title('Two Peaks of Alignment at Both Rings', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nObservations:")
print("  - Two distinct peaks in nematic order at y = 40 and 80 μm")
print("  - Both rings show perpendicular alignment to flow")
print("  - Central region (y = 60 μm) has lower order (competing flows)")
print("\n  This model predicts that symmetric division would produce")
print("  TWO aligned actin rings, as observed in some cell types!")

### Extension 2: Surface Contraction Wave

Some cells show **traveling waves** of contraction (e.g., oocyte surface contraction waves in *Xenopus*).

In [ ]:
print("=" * 60)
print("EXTENSION 2: SURFACE CONTRACTION WAVE")
print("=" * 60)

# Create simulation with traveling wave
sim_wave = ActiveNematicSimulation(
    Lx=100, Ly=100, dx=1.0, dt=0.01,
    flow_alignment=1.5,
    turnover_time=5.0,  # faster turnover for dynamic waves
    elastic_constant=2.0
)

# Initialize with wave flow (will update each step)
sim_wave.set_flow_field('contraction_wave', flow_strength=1.5)

print("\nSimulating traveling contraction wave...")
print("Wave propagates in x-direction")
print("This will take a bit longer...\n")

# Run with snapshots
n_steps = int(50.0 / sim_wave.dt)
snapshot_times = [0, 10, 20, 30, 40]
snapshots = []

for i in range(n_steps):
    # Update flow field (wave propagates)
    if i % 10 == 0:
        sim_wave.set_flow_field('contraction_wave', flow_strength=1.5)
    
    sim_wave.step()
    
    if sim_wave.t in snapshot_times:
        S_field = np.sqrt(sim_wave.Qxx**2 + sim_wave.Qxy**2)
        snapshots.append((sim_wave.t, S_field.copy(), sim_wave.Qxx.copy()))
        print(f"  Snapshot at t = {sim_wave.t:.1f} s")

print("\nSimulation complete!")

In [ ]:
# Visualize wave snapshots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (t, S_field, Qxx) in enumerate(snapshots):
    if idx < 6:
        im = axes[idx].imshow(S_field, extent=[0, sim_wave.Lx, 0, sim_wave.Ly],
                             origin='lower', cmap='viridis', vmin=0, vmax=0.4)
        axes[idx].set_title(f't = {t:.0f} s', fontsize=12)
        axes[idx].set_xlabel('x (μm)')
        axes[idx].set_ylabel('y (μm)')
        plt.colorbar(im, ax=axes[idx], label='S')

# Remove extra subplot
if len(snapshots) < 6:
    fig.delaxes(axes[-1])

plt.suptitle('Traveling Contraction Wave: Nematic Order', fontsize=16)
plt.tight_layout()
plt.show()

print("\nObservations:")
print("  - Wave of high nematic order propagates in x-direction")
print("  - Alignment follows the wave front")
print("  - Behind the wave, turnover randomizes filaments again")
print("\n  This demonstrates that the flow-alignment mechanism can")
print("  organize filaments in TRAVELING WAVES, not just static rings!")

---

## Summary and Discussion

### What We've Learned

1. **Active nematic gel theory** accurately describes actin filament alignment during cytokinesis

2. **Flow-alignment coupling** is the key mechanism:
   - Compressive flow → filaments align perpendicular to flow
   - Balanced by actin turnover and elastic interactions

3. **Quantitative predictions** match experimental observations:
   - Time scale of alignment (~10-20 s)
   - Spatial localization to convergence zones
   - Dependence on flow strength and turnover rate

4. **Extensions** show the model is general:
   - Symmetric division → two aligned rings
   - Contraction waves → traveling alignment patterns

### Biological Implications

- **Mechanical self-organization**: No specific "template" needed - flow geometry determines ring position
- **Robustness**: Multiple processes (flow, alignment, turnover) ensure reliable furrow formation
- **Scalability**: Same mechanism could work across different cell sizes and shapes

### Model Limitations and Future Directions

1. **Active stress feedback**: Real actin generates forces that affect the flow (not included here)
2. **3D geometry**: Embryo is a 3D ellipsoid, not flat 2D surface
3. **Myosin regulation**: RhoA and other regulators control myosin activity dynamically
4. **Discrete filaments**: Q-tensor is continuum approximation of discrete, fluctuating filaments

### Comparison to Reymann et al. (2016)

Our implementation captures the **essential physics** of the Reymann model:

| Feature | Reymann et al. | Our Implementation |
|---------|----------------|--------------------|
| Q-tensor evolution | ✓ | ✓ |
| Flow-alignment | ✓ | ✓ |
| Cortical flow | ✓ (from PIV data) | ✓ (prescribed) |
| 2D geometry | ✓ (embryo surface) | ✓ (flat domain) |
| Parameter fitting | ✓ (from experiments) | ✓ (literature values) |
| Active stress feedback | ✓ | ✗ (could add) |
| 3D shape changes | ✓ | ✗ (2D only) |

### References

- **Reymann et al. (2016)** "Cortical flow aligns actin filaments to form a furrow." *eLife* 5:e17807
- **Salbreux et al. (2009)** "Hydrodynamics of cellular cortical flows..." *Phys. Rev. Lett.* 103:058102
- **Kruse et al. (2005)** "Generic theory of active polar gels..." *Eur. Phys. J. E* 16:5
- **Prost et al. (2015)** "Active gel physics." *Nature Physics* 11:111

---

## Try It Yourself!

Modify the code above to explore:
- Different flow geometries
- Parameter sensitivities
- Other biological scenarios (wound healing, morphogenesis, etc.)

The `reymann_simulation.py` module provides a flexible framework for these explorations!